# 🎨 Google Photos AI 超能工坊 (Private AI Photo Studio)

本筆記本直接銜接 `Organized_Photos/` 整理相簿，**100% 在 Google 雲端 GPU 執行**，不消耗本機電腦效能！

### 核心三大超能力：
1. **👤 人臉識別與自動人物分群**：媲美 Google 相簿，InsightFace 深度特徵 + DBSCAN 聚類，自動歸納核心人物。
2. **🔍 繁體中文語義搜圖**：Multilingual CLIP 向量檢索，不用標籤，打字「看海的背影」、「生日吹蠟燭」0.05 秒精準出圖。
3. **🌐 雙軌 HTML 體驗**：內建 Gradio 現代 HTML5 操作面板（支援手機連線），並可一鍵匯出單檔 `Family_Album.html` 離線相簿！

> **💡 建議**：請先至 Colab 選單「執行階段」➜「變更執行階段類型」，確認硬體加速器已選擇 **GPU (T4 / L4 / A100)**。

## 步驟 1：掛載 Google 雲端硬碟 (Google Drive)

In [ ]:
from google.colab import drive
import os, sys, shutil, glob, json, hashlib, base64, io
from pathlib import Path

drive.mount('/content/drive')
print('✓ Google Drive 掛載成功！')

## 步驟 2：安裝核心 AI 與現代 HTML5 介面套件

In [ ]:
# 安裝 Gradio 現代 Web 面板、Sentence-Transformers、InsightFace 與圖像支援（鎖定 pillow<13.0 相容 Gradio）
!pip install -q gradio sentence-transformers insightface onnxruntime-gpu "pillow>=10.0,<13.0" pillow-heif scikit-learn
import pillow_heif
pillow_heif.register_heif_opener()
print('✓ 核心 AI 函式庫安裝完成（已啟用 iPhone HEIC/HEIF 解碼支援）！')

## 步驟 3：目錄設定與相片掃描（自動排除 Duplicates 重複區）

In [ ]:
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None

DRIVE_ROOT = Path('/content/drive/MyDrive')
PHOTOS_DIR = DRIVE_ROOT / 'Organized_Photos'
ARCHIVE_DIR = PHOTOS_DIR / 'Photos_Archive' if (PHOTOS_DIR / 'Photos_Archive').exists() else PHOTOS_DIR
INDEX_DIR = DRIVE_ROOT / 'Photo_AI_Index'
RESTORED_DIR = DRIVE_ROOT / 'Restored_Photos'

INDEX_DIR.mkdir(parents=True, exist_ok=True)
RESTORED_DIR.mkdir(parents=True, exist_ok=True)

# 若已有新歸檔相片想重算全相簿人臉群組，將下方改為 True
FORCE_RECLUSTER_FACES = False

# 支援的圖片格式（排除 Duplicates 重複隔離區、Restored 修復區與隱藏暫存檔）
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.heic', '.heif'}
all_photos = [
    p for p in ARCHIVE_DIR.rglob('*')
    if p.is_file()
    and p.suffix.lower() in VALID_EXTS
    and not p.name.startswith('.')
    and 'Duplicates' not in p.parts
    and 'Restored' not in p.parts
]

print(f'相簿來源目錄：{ARCHIVE_DIR}')
print(f'已載入有效相片總數：{len(all_photos)} 張')
if len(all_photos) == 0:
    print('⚠️ 目前 Organized_Photos 目錄下尚無相片，請先確認第一本相簿整理筆記本已完成。')

## 步驟 4：建立「繁體中文語義搜圖」向量索引
使用 Multilingual-CLIP 多模態模型，將每張照片編碼為 512 維語義特徵向量。
（支援增量更新：既有照片秒級載入，僅針對新增照片計算特徵）

In [ ]:
import torch
import numpy as np
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None
import pillow_heif
pillow_heif.register_heif_opener()
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'目前使用運算裝置：{device.upper()}')

clip_model = SentenceTransformer('clip-ViT-B-32-multilingual-v1', device=device)

clip_index_file = INDEX_DIR / 'clip_embeddings.npy'
clip_paths_file = INDEX_DIR / 'clip_paths.json'

cached_map = {}
if clip_index_file.exists() and clip_paths_file.exists():
    try:
        old_embs = np.load(clip_index_file)
        old_paths = json.loads(clip_paths_file.read_text(encoding='utf-8'))
        if len(old_embs) == len(old_paths):
            cached_map = {p: emb for p, emb in zip(old_paths, old_embs)}
            print(f'✓ 已讀取既有語義快取：{len(cached_map)} 筆')
    except Exception as e:
        print(f'快取重設：{e}')

photo_paths = []
embeddings_list = []
new_photos = []

for p in all_photos:
    p_str = str(p)
    if p_str in cached_map:
        photo_paths.append(p)
        embeddings_list.append(cached_map[p_str].reshape(1, -1))
    else:
        new_photos.append(p)

failed_clip = 0
if new_photos:
    print(f'開始計算新增相片語義特徵向量：共 {len(new_photos)} 張...')
    batch_size = 32
    for i in range(0, len(new_photos), batch_size):
        batch_paths = new_photos[i:i+batch_size]
        batch_images = []
        valid_batch_paths = []
        for p in batch_paths:
            try:
                with Image.open(p) as im:
                    im_fixed = ImageOps.exif_transpose(im).convert('RGB')
                    im_fixed.thumbnail((1024, 1024))
                    batch_images.append(im_fixed)
                valid_batch_paths.append(p)
            except Exception:
                failed_clip += 1
                continue
        if batch_images:
            with torch.no_grad():
                embs = clip_model.encode(batch_images, convert_to_numpy=True, normalize_embeddings=True)
            embeddings_list.append(embs)
            photo_paths.extend(valid_batch_paths)
        if (i // batch_size) % 10 == 0:
            print(f'進度：{min(i + batch_size, len(new_photos))}/{len(new_photos)} 張...')

if embeddings_list:
    photo_embeddings = np.vstack(embeddings_list)
    np.save(clip_index_file, photo_embeddings)
    clip_paths_file.write_text(json.dumps([str(p) for p in photo_paths], ensure_ascii=False), encoding='utf-8')
    print(f'✓ 語義索引就緒！共計 {len(photo_paths)} 張' + (f'（跳過損毀或無法解碼 {failed_clip} 張）' if failed_clip else '。'))
else:
    photo_embeddings = np.zeros((0, 512), dtype=np.float32)
    print('⚠️ 無有效相片可建立索引。')

## 步驟 5：建立「人臉辨識與自動人物分群」索引
使用 InsightFace 辨識人臉，並透過 DBSCAN 聚類自動歸納核心人物群組。

In [ ]:
from sklearn.cluster import DBSCAN
import insightface
from insightface.app import FaceAnalysis

faces_cluster_file = INDEX_DIR / 'people_clusters.json'

if faces_cluster_file.exists() and not globals().get('FORCE_RECLUSTER_FACES', False):
    print('✓ 發現既有人物分群檔案，直接載入中（若想納入新整理相片，請將步驟 3 的 FORCE_RECLUSTER_FACES 設為 True）...')
    with open(faces_cluster_file, 'r', encoding='utf-8') as f:
        people_clusters = json.load(f)
    print(f'✓ 已載入 {len(people_clusters)} 組人物相簿！')
else:
    print('初始化 InsightFace 人臉辨識引擎 (buffalo_l)...')
    face_app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
    face_app.prepare(ctx_id=0, det_size=(640, 640))

    face_vectors = []
    face_records = []

    print('開始掃描相簿人臉...')
    for idx, p in enumerate(all_photos):
        try:
            with Image.open(p) as im:
                img = ImageOps.exif_transpose(im).convert('RGB')
                img.thumbnail((1600, 1600))
            img_np = np.array(img)[:, :, ::-1]
            faces = face_app.get(img_np)
            for f in faces:
                norm_emb = f.embedding / np.linalg.norm(f.embedding)
                face_vectors.append(norm_emb)
                face_records.append({'path': str(p), 'bbox': [float(x) for x in f.bbox]})
        except Exception:
            continue
        if idx % 200 == 0 and idx > 0:
            print(f'人臉掃描進度：{idx}/{len(all_photos)}...')

    print(f'✓ 人臉掃描完成，累計偵測到 {len(face_vectors)} 張人臉！進行 DBSCAN 密度聚類...')
    if len(face_vectors) > 0:
        clustering = DBSCAN(eps=0.55, min_samples=3, metric='cosine')
        labels = clustering.fit_predict(np.array(face_vectors))

        people_clusters = {}
        for label, rec in zip(labels, face_records):
            if label == -1:
                continue
            key = f'人物_{label+1:02d}'
            if key not in people_clusters:
                people_clusters[key] = []
            if rec['path'] not in people_clusters[key]:
                people_clusters[key].append(rec['path'])

        with open(faces_cluster_file, 'w', encoding='utf-8') as f:
            json.dump(people_clusters, f, ensure_ascii=False, indent=2)
        print(f'🎉 人物分群成功！共聚類出 {len(people_clusters)} 位主要人物。')
    else:
        people_clusters = {}

## 步驟 6：啟動 Gradio 現代 HTML5 互動相簿工作台
點擊下方產生的網址，即可在瀏覽器（桌面與手機均可）享受秒級 AI 搜圖與人物相簿！

In [ ]:
import inspect
import gradio as gr

def _to_gallery_item(p: Path):
    """瀏覽器不原生支援 HEIC/HEIF，遇蘋果格式時即時轉為預覽縮圖，其餘直接回傳檔案路徑。"""
    if p.suffix.lower() in ('.heic', '.heif'):
        with Image.open(p) as im:
            rgb = ImageOps.exif_transpose(im).convert('RGB')
            rgb.thumbnail((1280, 1280))
            return rgb
    return str(p)

def search_photos(query_text, top_k=18):
    if not query_text or not query_text.strip() or len(photo_paths) == 0:
        return []
    with torch.no_grad():
        q_emb = clip_model.encode([query_text], convert_to_numpy=True, normalize_embeddings=True)[0]
    scores = np.dot(photo_embeddings, q_emb)
    top_indices = np.argsort(scores)[::-1][:int(top_k) * 2]
    results = []
    for idx in top_indices:
        p = photo_paths[idx]
        if not p.exists():
            continue
        score = scores[idx]
        caption = f"{p.name} (相似度: {score:.2f})"
        try:
            results.append((_to_gallery_item(p), caption))
        except Exception:
            continue
        if len(results) >= int(top_k):
            break
    return results

def filter_by_person(person_name):
    if not person_name or person_name not in people_clusters:
        return []
    photos = people_clusters[person_name]
    results = []
    for p_str in photos[:30]:
        p = Path(p_str)
        if not p.exists():
            continue
        try:
            results.append((_to_gallery_item(p), p.name))
        except Exception:
            continue
    return results

theme = gr.themes.Soft(primary_hue="teal", neutral_hue="slate")
blocks_kwargs = {"title": "AI 私有相簿超能工坊"}
launch_kwargs = {"share": True, "debug": False}
if "theme" in inspect.signature(gr.Blocks.__init__).parameters:
    blocks_kwargs["theme"] = theme
elif "theme" in inspect.signature(gr.Blocks.launch).parameters:
    launch_kwargs["theme"] = theme

with gr.Blocks(**blocks_kwargs) as demo:
    gr.Markdown("# 📷 Google Photos AI 私有相簿超能工坊")
    gr.Markdown("直接連結 Google One 2TB 相簿，以開源 AI 模型進行語義搜圖與人物瀏覽。")

    with gr.Tabs():
        with gr.TabItem("🔍 繁體中文自然語言搜圖"):
            with gr.Row():
                query_input = gr.Textbox(placeholder="輸入自然語言，如：在海邊看夕陽、穿紅色洋裝、生日吹蠟燭、夜市吃小吃...", label="搜尋描述", scale=4)
                k_slider = gr.Slider(minimum=6, maximum=36, value=18, step=6, label="顯示張數", scale=1)
                search_btn = gr.Button("立即搜尋 ⚡", variant="primary", scale=1)

            search_gallery = gr.Gallery(label="搜尋結果 (點擊可放大檢視)", columns=3, height=600)
            search_btn.click(fn=search_photos, inputs=[query_input, k_slider], outputs=search_gallery)
            query_input.submit(fn=search_photos, inputs=[query_input, k_slider], outputs=search_gallery)

        with gr.TabItem("👤 人物智慧相簿"):
            person_keys = list(people_clusters.keys())
            with gr.Row():
                person_dropdown = gr.Dropdown(choices=person_keys, value=person_keys[0] if person_keys else None, label="選擇人物", scale=3)
                filter_btn = gr.Button("查看相片 📁", variant="primary", scale=1)

            person_gallery = gr.Gallery(label="該人物相片集", columns=3, height=600)
            filter_btn.click(fn=filter_by_person, inputs=person_dropdown, outputs=person_gallery)
            person_dropdown.change(fn=filter_by_person, inputs=person_dropdown, outputs=person_gallery)

    gr.Markdown("---\n*所有運算 100% 於 Google Colab 私有環境執行，相片資料零外洩。*")

demo.launch(**launch_kwargs)

## 步驟 7：一鍵匯出單檔 `Family_Album.html` (內嵌縮圖免開 Colab 離線網頁)
將人物分群與精華相片縮圖直接內嵌匯出為單一純 HTML 網頁，日後用電腦或手機雙擊即可秒開瀏覽！

In [ ]:
import html as html_lib

album_html_file = DRIVE_ROOT / 'Family_Album.html'

html_content = '''<!DOCTYPE html>
<html lang="zh-TW">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>家族回憶錄 - 私有智慧相簿</title>
  <style>
    body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; background: #f8fafc; color: #1e293b; margin: 0; padding: 24px; }
    h1 { font-size: 24px; margin-bottom: 8px; color: #0f766e; }
    .person-card { background: white; border-radius: 12px; padding: 16px; margin-bottom: 24px; box-shadow: 0 2px 8px rgba(0,0,0,0.05); }
    .person-title { font-size: 18px; font-weight: bold; margin-bottom: 12px; color: #0f172a; }
    .photo-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(180px, 1fr)); gap: 12px; }
    .photo-item { border-radius: 8px; overflow: hidden; background: #e2e8f0; display: flex; flex-direction: column; font-size: 12px; color: #475569; text-align: center; }
    .photo-item img { width: 100%; aspect-ratio: 1; object-fit: cover; display: block; }
    .photo-caption { padding: 6px 8px; word-break: break-all; background: #f1f5f9; }
  </style>
</head>
<body>
  <h1>📷 我的私有家庭相簿畫廊</h1>
  <p>由 Colab InsightFace 自動完成人臉聚類，內嵌精華預覽縮圖。</p>
'''

for person, p_list in people_clusters.items():
    safe_person = html_lib.escape(str(person))
    html_content += f'<div class="person-card">\n<div class="person-title">👤 {safe_person} (共 {len(p_list)} 張照片)</div>\n<div class="photo-grid">\n'
    for path_str in p_list[:12]:
        p_obj = Path(path_str)
        fname = html_lib.escape(p_obj.name)
        thumb_tag = ''
        try:
            with Image.open(p_obj) as im:
                im_rgb = ImageOps.exif_transpose(im).convert('RGB')
                im_rgb.thumbnail((320, 320))
                buf = io.BytesIO()
                im_rgb.save(buf, format='JPEG', quality=75)
                b64 = base64.b64encode(buf.getvalue()).decode('ascii')
                thumb_tag = f'<img src="data:image/jpeg;base64,{b64}" alt="{fname}" loading="lazy">'
        except Exception:
            pass
        html_content += f'<div class="photo-item">{thumb_tag}<div class="photo-caption">{fname}</div></div>\n'
    html_content += '</div>\n</div>\n'

html_content += '''
</body>
</html>
'''

album_html_file.write_text(html_content, encoding='utf-8')
print(f'✨ 已生成單檔離線相簿網頁：{album_html_file}')
print('日後在 Google Drive 下載或直接點開該 HTML 即可秒看人物畫廊！')